# Configuration

This notebook generates the table for differences by configuration.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
from typing import Dict, Tuple
from depsurf import DepKind, DiffResult, IssueEnum, VersionGroup
from utils import (
    load_pkl,
    GRAY_DASH,
    save_latex,
    FLAVOR_NAMES,
    ARCH_NAMES,
    mini_bar,
)

data: DiffResult = load_pkl("config")

ISSUES = {
    IssueEnum.NEW: "#",
    IssueEnum.ADD: "$+$",
    IssueEnum.REMOVE: "$-$",
    IssueEnum.CHANGE: r"$\\Delta$",
}

KINDS = {
    DepKind.STRUCT: "Struct",
    DepKind.CONFIG: "Config",
    DepKind.FUNC: "Func",
    DepKind.TRACEPOINT: "Tracept",
    # DepKind.TRACEPOINT: r"\\shortstack{Tracept\\\\($\\Delta=0$)}",
    DepKind.SYSCALL: r"\\shortstack{Native\\\\Syscall}",
    DepKind.REGISTER: "Register",
}

ROWS = [
    (DepKind.CONFIG, IssueEnum.NEW),
    *(
        (kind, issue)
        for kind in (DepKind.FUNC, DepKind.STRUCT, DepKind.TRACEPOINT)
        for issue in ISSUES
    ),
    *(
        (DepKind.SYSCALL, issue)
        for issue in [i for i in ISSUES if i != IssueEnum.CHANGE]
    ),
    (DepKind.REGISTER, IssueEnum.CHANGE),
]

GROUPS = {
    VersionGroup.ARCH: "Architecture",
    VersionGroup.FLAVOR: "Flavor",
}

DEFAULT = (r"\\multicolumn{1}{c|}{def-}", r"\\multicolumn{1}{c|}{ault}")

COLORS = {
    DepKind.FUNC: "func",
    DepKind.STRUCT: "struct",
    DepKind.TRACEPOINT: "tp",
    DepKind.SYSCALL: "syscall",
}


def format_val(v) -> str:
    if v > 1000:
        return f"{v / 1000:.1f}k"
    if v == 0:
        return GRAY_DASH
    return str(v)


table: Dict[Tuple[str, str], Dict[Tuple[DepKind, IssueEnum], str]] = {}

def_pair_result = list(data[VersionGroup.ARCH].values())[0]
col = {(kind, issue): GRAY_DASH for kind, issue in ROWS}
for kind, kind_result in def_pair_result.iter_kinds():
    for issue, count in kind_result.iter_issues():
        if issue == IssueEnum.OLD:
            col[(kind, IssueEnum.NEW)] = format_val(count)
            continue
table[DEFAULT] = col

max_counts = {
    kind: max(
        count
        for group, group_result in data.items()
        for pair, pair_result in group_result.items()
        for issue, count in pair_result.kind_results[kind].iter_issues()
        if issue not in (IssueEnum.OLD, IssueEnum.NEW)
    )
    for kind in [DepKind.FUNC, DepKind.STRUCT, DepKind.TRACEPOINT, DepKind.SYSCALL]
}

for group, group_result in data.items():
    if group not in GROUPS:
        continue
    for pair, pair_result in group_result.items():
        col = {(kind, issue): GRAY_DASH for kind, issue in ROWS}
        for kind, kind_result in pair_result.iter_kinds():
            for issue, count in kind_result.iter_issues():
                if (kind, issue) not in ROWS:
                    continue
                text = format_val(count)
                if issue != IssueEnum.NEW:
                    text = mini_bar(
                        text,
                        count / max_counts[kind],
                        0.8,
                        COLORS[kind],
                    )
                col[(kind, issue)] = text
        if group == VersionGroup.ARCH:
            name = ARCH_NAMES[pair.v2.arch]
            col[(DepKind.REGISTER, IssueEnum.CHANGE)] = "Yes"
        elif group == VersionGroup.FLAVOR:
            name = FLAVOR_NAMES[pair.v2.flavor]
        table[(GROUPS[group], name)] = col


df = pd.DataFrame(
    {
        (group, name): {
            (KINDS[kind], ISSUES[issue]): text for (kind, issue), text in col.items()
        }
        for (group, name), col in table.items()
    }
)

latex = df.to_latex(
    multicolumn_format="c|", column_format=r"p{7ex}p{1.5ex}|r|rrrr|rrrr", multirow=True
)

save_latex(latex, "cfg", rotate=False)
df

In [ ]:
from pathlib import Path
import csv
import json

results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

table_counts: Dict[Tuple[str, str], Dict[Tuple[DepKind, IssueEnum], int | None]] = {}

def_pair_result = list(data[VersionGroup.ARCH].values())[0]
baseline = {(kind, issue): None for (kind, issue) in ROWS}
for kind, kind_result in def_pair_result.iter_kinds():
    for issue, count in kind_result.iter_issues():
        if issue == IssueEnum.OLD and (kind, IssueEnum.NEW) in baseline:
            baseline[(kind, IssueEnum.NEW)] = int(count)
table_counts[DEFAULT] = baseline

for group, group_result in data.items():
    if group not in GROUPS:
        continue
    for pair, pair_result in group_result.items():
        col = {(kind, issue): None for (kind, issue) in ROWS}
        for kind, kind_result in pair_result.iter_kinds():
            for issue, count in kind_result.iter_issues():
                if (kind, issue) not in col:
                    continue
                col[(kind, issue)] = int(count)
        if group == VersionGroup.ARCH:
            name = ARCH_NAMES[pair.v2.arch]
        else:
            name = FLAVOR_NAMES[pair.v2.flavor]
        if (DepKind.REGISTER, IssueEnum.CHANGE) in col:
            col[(DepKind.REGISTER, IssueEnum.CHANGE)] = None
        table_counts[(GROUPS[group], name)] = col

row_labels = [(KINDS[k], ISSUES[i]) for (k, i) in ROWS]
col_labels = list(table_counts.keys())
with (results_dir / "39_config.labels.json").open("w") as f:
    json.dump({"rows": [list(x) for x in row_labels], "cols": [list(x) for x in col_labels]}, f, indent=2)

import re

def _sanitize_token(s: str) -> str:
    s = str(s)
    s = s.replace("\\", " ")
    s = re.sub(r"\\[A-Za-z]+", "", s)
    s = s.replace("{", "").replace("}", "")
    s = s.replace("$", "")
    s = s.strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^0-9A-Za-z_]+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_").lower()

_KIND_PLAIN = {
    DepKind.CONFIG: "config",
    DepKind.FUNC: "func",
    DepKind.STRUCT: "struct",
    DepKind.TRACEPOINT: "tracept",
    DepKind.SYSCALL: "native_syscall",
    DepKind.REGISTER: "register",
}

_ISSUE_PLAIN = {
    IssueEnum.NEW: "no",
    IssueEnum.ADD: "plus",
    IssueEnum.REMOVE: "minus",
    IssueEnum.CHANGE: "delta"
}

def _row_id(kind: DepKind, issue: IssueEnum) -> str:
    k = _KIND_PLAIN.get(kind, _sanitize_token(KINDS[kind]))
    i = _ISSUE_PLAIN.get(issue, _sanitize_token(ISSUES[issue]))
    return f"{k}_{i}"

def _col_id(k: Tuple[str, str]) -> str:
    if k == DEFAULT:
        return "default"
    g, n = k
    g = _sanitize_token(g)
    n = _sanitize_token(n)
    if g and n:
        return f"{g}_{n}"
    return g or n or "col"

flat_col_headers = [_col_id(k) for k in col_labels]
flat_row_labels = [_row_id(kind, issue) for (kind, issue) in ROWS]

csv_path = results_dir / "39_config.csv"
with csv_path.open("w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["row"] + flat_col_headers)
    for (kind, issue), row_name in zip(ROWS, flat_row_labels):
        row = [row_name]
        for col_key in col_labels:
            v = table_counts[col_key].get((kind, issue), None)
            if v is None or v == 0:
                row.append("")
            else:
                row.append(str(int(v)))
        w.writerow(row)

print("Column headers (flattened, plain):")
print(flat_col_headers)
print("Row labels (flattened, plain):")
print(flat_row_labels)
print(f"Wrote {csv_path} and {results_dir / '39_config.labels.json'}")
